# 1. Setup

3D adds a CLIP text alignment loss to the fused representation in 3Cv1. The experiment uses four unfrozen blocks per tower, matching the published 3Cv1 depth. Inspired by [TDC](https://doi.org/10.3390/electronics15081553) and [pairwise feature learning](https://ir.unimas.my/id/eprint/43065/2/PAIRWISE.pdf). This is a compositional adaptation with pair prompts and fixed weights.

In [ ]:
import gc, json, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from timm.layers import set_reentrant_ckpt
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3d_dino_semantic_pairwise_fusion', 'D'
DEBUG = True
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          unfreeze_blocks=[4], backbone_lr=1e-5,
          crop_lr=1e-4, disease_lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256, feature_dim=128,
          dino_revision='main', fusion_attention_heads=4,
          primitive_weight_start=0.75, primitive_weight_end=0.25,
          text_revision='main', text_batch_size=16, semantic_weight=0.25,
          semantic_scale=10.0, semantic_mix=0.25,
          pair_prompt='a photo of a {crop} leaf with {disease}',
          healthy_prompt='a photo of a healthy {crop} leaf')

EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 0  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))

# 2. Data

Use the shared train split, transforms and training-derived pair labels.

In [ ]:
# Same common data path as the old CLIP notebook; DINO uses imagenet normalization.
frames = [load_split(cfg.TRAIN_CSV)]
assert len(frames[0]) == 8000
pairs = get_pairs()
assert pairs.shape == (cfg.NUM_SEEN_PAIRS, 2)
def loaders():
    cfg.set_seed(cfg.SEED)
    return get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
train_loader, val_loader, seen_loader, unseen_loader = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
assert not set(train_loader.dataset.frame.image) & set(val_loader.dataset.frame.image)
sample_counts = dict(train=len(train_loader.dataset), val=len(val_loader.dataset))
print('Device:', cfg.DEVICE, 'Data:', cfg.DATA_DIR, 'Samples:', sample_counts)
concept_names = get_crop_and_disease_names()

# 3. Model

Keep the original residual fusion, primitive heads and observed-pair recomposition. Train an extra projection against frozen CLIP pair prompts using real training images. The 0.25 semantic loss weight and score mix are fixed for this initial experiment.

In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self, depth):
        self.requires_grad_(False)
        for tower in (self.crop, self.disease):
            tower.blocks[-depth:].requires_grad_(True)
            tower.set_grad_checkpointing(True)

    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))


RECIPE = dict(synthetic_pairs='observed_training_only', loss='moving_weighted_sum', semantic='clip_pair_alignment')

class Model(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        width, hidden, dim = backbone.width, HP['hidden_dim'], HP['feature_dim']
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)
        self.crop_aux = nn.Linear(dim, cfg.NUM_CROPS)
        self.disease_aux = nn.Linear(dim, cfg.NUM_DISEASES)
        self.attention = nn.MultiheadAttention(dim, HP['fusion_attention_heads'], batch_first=True)
        self.image_fusion = nn.Sequential(nn.LayerNorm(2 * dim), nn.Linear(2 * dim, dim), nn.GELU())
        self.crop_embedding = nn.Embedding(cfg.NUM_CROPS, dim)
        self.disease_embedding = nn.Embedding(cfg.NUM_DISEASES, dim)
        self.text_projection = nn.Linear(dim, 512)
        self.register_buffer('pair_text', torch.empty(0, 512), persistent=False)
        self.pair_fusion = nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU())

    def features(self, images):
        crop, disease = self.backbone(images)
        return self.crop_projection(crop), self.disease_projection(disease)

    def fuse(self, crop, disease):
        combined = crop + disease
        tokens = torch.stack((crop + combined, disease + combined), dim=1)
        attended, _ = self.attention(tokens, tokens, tokens, need_weights=False)
        residuals = torch.stack((disease, crop), dim=1)
        return self.image_fusion((attended + residuals).flatten(1))

    def score(self, crop, disease, candidate_pairs):
        image = self.fuse(crop, disease)
        crop_ids, disease_ids = candidate_pairs.unbind(1)
        candidates = self.pair_fusion(torch.cat(
            (self.crop_embedding(crop_ids), self.disease_embedding(disease_ids)), dim=1))
        return image @ candidates.T


    def semantic_logits(self, crop, disease):
        image = F.normalize(self.text_projection(self.fuse(crop, disease)), dim=1)
        return HP['semantic_scale'] * (image @ self.pair_text.T)


def sample_observed_pairs(crops, diseases, pair_rows, generator):
    """Sample at most one batch's worth of admissible cross-image combinations."""
    rows = pair_rows[crops[:, None], diseases[None, :]]
    eligible = (rows >= 0) & ~torch.eye(len(crops), dtype=torch.bool, device=rows.device)
    crop_sources, disease_sources = eligible.nonzero(as_tuple=True)
    order = torch.randperm(len(crop_sources), generator=generator)[:len(crops)].to(rows.device)
    crop_sources, disease_sources = crop_sources[order], disease_sources[order]
    return crop_sources, disease_sources, rows[crop_sources, disease_sources]


def training_loss(model, images, labels, candidates, pair_rows, generator, primitive_weight):
    crop, disease = model.features(images)
    crops, diseases = candidates[labels].unbind(1)
    smoothing = HP['label_smoothing']
    primitive = (F.cross_entropy(model.crop_aux(crop), crops, label_smoothing=smoothing)
                 + F.cross_entropy(model.disease_aux(disease), diseases, label_smoothing=smoothing)) / 2
    joint = F.cross_entropy(model.score(crop, disease, candidates), labels, label_smoothing=smoothing)
    src_crop, src_disease, targets = sample_observed_pairs(crops, diseases, pair_rows, generator)
    if targets.numel():
        synthetic = model.score(crop[src_crop], disease[src_disease], candidates)
        joint = (joint + F.cross_entropy(synthetic, targets, label_smoothing=smoothing)) / 2
    semantic = F.cross_entropy(model.semantic_logits(crop, disease), labels, label_smoothing=smoothing)
    value = primitive_weight * primitive + (1 - primitive_weight) * joint
    return value + HP['semantic_weight'] * semantic, targets.numel()


class TextOnly:
    """No CLIPModel, CLIPVisionModel, OpenCLIP model, or image encoder is created."""
    def __init__(self,config,device,allow_download=False,revision=None):
        from transformers import CLIPTextModelWithProjection, CLIPTokenizerFast
        path=snapshot('openai/clip-vit-base-patch16',revision or config['text_revision'],allow_download,
            ['*.json','vocab.json','merges.txt','model.safetensors','pytorch_model.bin'])
        self.model=CLIPTextModelWithProjection.from_pretrained(path,local_files_only=True).to(device).eval().requires_grad_(False)
        self.tokenizer=CLIPTokenizerFast.from_pretrained(path,local_files_only=True)
        if any('vision' in name for name,_ in self.model.named_modules()):raise RuntimeError('A CLIP vision encoder was instantiated.')
        self.dimension=self.model.config.projection_dim;self.device=device
        if self.dimension!=512:raise ValueError('Expected the CLIP B/16 512-dimensional text projection.')
        weight=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='openai/clip-vit-base-patch16',revision=path.name,
            class_name=type(self.model).__name__,dimension=self.dimension,trainable_parameters=0,clip_image_encoder=False)
    @torch.no_grad()
    def encode(self,prompts,size):
        self.assert_frozen();result=[]
        for start in range(0,len(prompts),size):
            tokens=self.tokenizer(prompts[start:start+size],padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
            result.append(F.normalize(self.model(**tokens).text_embeds.float(),dim=1))
        return torch.cat(result)
    def assert_frozen(self):
        if self.model.training or any(p.requires_grad or p.grad is not None for p in self.model.parameters()):
            raise RuntimeError('CLIP text encoder must be eval, frozen and gradient-free.')


def pair_text_bank(text, candidates, names):
    crops, diseases = names
    prompts = []
    for crop_id, disease_id in candidates.tolist():
        crop = crops[crop_id].replace('_', ' ')
        disease = diseases[disease_id].replace('_', ' ')
        template = HP['healthy_prompt'] if disease.strip().casefold() == 'healthy' else HP['pair_prompt']
        prompts.append(template.format(crop=crop, disease=disease))
    return text.encode(prompts, HP['text_batch_size'])


def make_model(depth, metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.policy(depth)
    text = TextOnly(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('text', {}).get('revision'))
    cfg.set_seed(cfg.SEED)
    model = Model(backbone).to(cfg.DEVICE)
    model.pair_text = pair_text_bank(text, pairs, concept_names)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, text, dict(dino=backbone.metadata, text=text.metadata)


def make_predict(model, candidate_pairs):
    candidates = candidate_pairs.to(cfg.DEVICE)

    @torch.no_grad()
    def predict(images):
        model.eval()
        crop, disease = model.features(images.to(cfg.DEVICE))
        joint = model.score(crop, disease, candidates).log_softmax(dim=1)
        semantic = model.semantic_logits(crop, disease).log_softmax(dim=1)
        return (1 - HP['semantic_mix']) * joint + HP['semantic_mix'] * semantic
    return predict

# 4. Training

Select the best epoch by shared validation Top-1 over training-derived candidates.

In [ ]:
def train_trial(depth, revisions, best, checkpoint_path):
    train_loader, val_loader, _, _ = loaders()
    train_loader.pin_memory = val_loader.pin_memory = False
    model, text, metadata = make_model(depth, revisions)
    del text
    candidates = pairs.to(cfg.DEVICE)
    pair_rows = torch.full((cfg.NUM_CROPS, cfg.NUM_DISEASES), -1, device=cfg.DEVICE)
    pair_rows[candidates[:, 0], candidates[:, 1]] = torch.arange(len(candidates), device=cfg.DEVICE)
    generator = torch.Generator().manual_seed(cfg.SEED)
    groups = {'backbone': [], 'crop': [], 'disease': []}
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            key = 'backbone' if name.startswith('backbone.') else 'disease' if 'disease' in name else 'crop'
            groups[key].append(parameter)
    optimizer = torch.optim.AdamW(
        [dict(params=groups[key], lr=HP[key + '_lr']) for key in groups],
        weight_decay=HP['weight_decay'])
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    predict = make_predict(model, pairs)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        model.backbone.eval()
        for tower in (model.backbone.crop, model.backbone.disease):
            tower.blocks[-depth:].train()
        progress = (epoch - 1) / (EPOCHS - 1) if EPOCHS > 1 else 0.5
        primitive_weight = HP['primitive_weight_start'] + progress * (
            HP['primitive_weight_end'] - HP['primitive_weight_start'])
        total_loss = synthetic_count = samples = 0
        for images, _, _, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            value, synthetic = training_loss(model, images.to(cfg.DEVICE), labels.to(cfg.DEVICE),
                                             candidates, pair_rows, generator, primitive_weight)
            if not torch.isfinite(value):
                raise RuntimeError('Nonfinite training loss.')
            value.backward()
            optimizer.step()
            samples += len(labels)
            total_loss += value.item() * len(labels)
            synthetic_count += synthetic
        schedule.step()
        validation = val_accuracy(predict, val_loader)
        record = dict(depth=depth, epoch=epoch, val_top1=validation, loss=total_loss / samples)
        record.update(synthetic_examples=synthetic_count, primitive_weight=primitive_weight)
        history.append(record)
        print(record)
        if best is None or validation > best['val_top1']:
            best = record
            torch.save(dict(identity, depth=depth, best_val=validation, best_epoch=epoch, pairs=pairs,
                            metadata=metadata, state_dict=model.state_dict()), checkpoint_path)
    return best, history, metadata

OUT.mkdir(parents=True, exist_ok=False)
set_reentrant_ckpt(False)  # The frozen prefix produces tensors without gradients.
identity = dict(model_name=MODEL_NAME, family=FAMILY, debug=DEBUG, hp=HP, recipe=RECIPE)
checkpoint_path = OUT / (MODEL_NAME + '.pt')
best, revisions, trials = None, None, []
for depth in HP['unfreeze_blocks']:
    best, history, revisions = train_trial(depth, revisions, best, checkpoint_path)
    trials.append(max(history, key=lambda record: record['val_top1']))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
selection = dict(identity, checkpoint=checkpoint_path.name, depth=best['depth'],
                 best_val=best['val_top1'], best_epoch=best['epoch'], trials=trials,
                 train_images=train_loader.dataset.frame.image.tolist(),
                 validation_images=val_loader.dataset.frame.image.tolist(),
                 development_pairs=pairs.tolist())
(OUT / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf8')
display(pd.DataFrame(trials))
print(f"Selected {best['depth']} blocks per tower at epoch {best['epoch']} ({best['val_top1']:.2f}% val Top-1).")

# 5. Evaluation

Build final prompts only after checkpoint validation and loading. Rank all 333 pairs with the same scoring rule.

In [ ]:
RUN_FINAL_EVALUATION = False  # Enable only after all model choices are frozen.
reports = []
if RUN_FINAL_EVALUATION:
    if DEBUG:
        raise ValueError('Official evaluation requires a full non-DEBUG run.')
    run_folder = OUT
    selection = json.loads((run_folder / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(run_folder / selection['checkpoint'], map_location='cpu', weights_only=True)
    expected = dict(model_name=MODEL_NAME, family=FAMILY, debug=False, hp=HP, recipe=RECIPE)
    if any(record.get(key) != value for record in (selection, checkpoint)
           for key, value in expected.items()):
        raise ValueError('The selected run or checkpoint does not match the frozen settings.')
    if selection['development_pairs'] != pairs.tolist() or not torch.equal(checkpoint['pairs'], pairs):
        raise ValueError('Development candidates changed.')
    if (selection['train_images'] != train_loader.dataset.frame.image.tolist()
            or selection['validation_images'] != val_loader.dataset.frame.image.tolist()):
        raise ValueError('Common train/validation membership changed.')
    if (checkpoint['depth'], checkpoint['best_epoch'], checkpoint['best_val']) != (
            selection['depth'], selection['best_epoch'], selection['best_val']):
        raise ValueError('Checkpoint does not match the validation winner.')
    model, text, metadata = make_model(checkpoint['depth'], checkpoint['metadata'])
    if metadata != checkpoint['metadata']:
        raise ValueError('Public pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    for tower in (model.backbone.crop, model.backbone.disease):
        tower.set_grad_checkpointing(False)
    # Only now open the official candidates and test loaders.
    final_pairs = get_pairs(final_evaluation=True)
    model.pair_text = pair_text_bank(text, final_pairs, concept_names)
    del text
    seen_loader, unseen_loader = get_final_test_loaders(
        HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    sample_counts.update(seen=len(seen_loader.dataset), unseen=len(unseen_loader.dataset))
    predict_final = make_predict(model, final_pairs)
    # common.evaluate predicts clean seen, then clean unseen, then robustness.
    seen_count = sample_counts['seen']
    clean_end = seen_count + sample_counts['unseen']
    unseen_batches, processed = [], 0
    def predict(images):
        global processed
        scores = predict_final(images)
        if seen_count <= processed < clean_end:
            unseen_batches.append(scores.float().cpu())
        processed += len(images)
        return scores
    RESULTS = cfg.RESULTS_DIR / run_folder.name
    RESULTS.mkdir(parents=True, exist_ok=False)
    previous_results = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        result = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, dict(HP, selected_depth=checkpoint['depth']), robust=True)
    finally:
        cfg.RESULTS_DIR = previous_results
    unseen_labels = torch.as_tensor(unseen_loader.dataset.frame.pair_row.to_numpy())
    crop_acc, disease_acc = concept_accuracy(torch.cat(unseen_batches), unseen_labels, final_evaluation=True)
    result.update(unseen_test_crop_acc=crop_acc, unseen_test_disease_acc=disease_acc,
                  debug=DEBUG, best_val=checkpoint['best_val'])
    reports.append(result)
    del predict_final, model, unseen_batches

# 6. Save

Keep the checkpoint, metrics and confusion files.

In [ ]:
if RUN_FINAL_EVALUATION:
    for result in reports:
        (RESULTS/(result['model']+'.json')).write_text(json.dumps(result, indent=2), encoding='utf8')
    summary = dict(selection=selection, results=reports, hp=HP,
                   environment=environment, samples=sample_counts)
    (RESULTS/(MODEL_NAME+'_comparison.json')).write_text(json.dumps(summary, indent=2), encoding='utf8')
    table = pd.DataFrame(reports)
    table.to_csv(RESULTS/(MODEL_NAME+'_comparison.csv'), index=False)
    display(table[['model','seen_top1','unseen_top1','harmonic_mean',
                   'unseen_test_crop_acc','unseen_test_disease_acc']])
    print('Checkpoints:', run_folder, 'Results:', RESULTS)
else:
    print('Save the complete checkpoint run folder after training:', OUT)